# 06 - Silver Layer

Mục tiêu:
- Đọc dữ liệu từ Bronze
- Làm sạch và chuẩn hóa
- Loại duplicate
- Kiểm tra khóa và quan hệ
- Ghi dữ liệu sạch sang Iceberg Silver tables


## 6.1. Tạo SparkSession


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ICEBERG_VERSION = "1.11.0"
HADOOP_VERSION = "3.4.2"

PACKAGES = (
    f"org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:{ICEBERG_VERSION},"
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION},"
    f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION}"
)

AWS_ACCESS_KEY_ID = (
    os.environ.get("AWS_ACCESS_KEY_ID")
    or os.environ.get("MINIO_ROOT_USER")
)

AWS_SECRET_ACCESS_KEY = (
    os.environ.get("AWS_SECRET_ACCESS_KEY")
    or os.environ.get("MINIO_ROOT_PASSWORD")
)

if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    raise RuntimeError(
        "Không tìm thấy credential MinIO trong biến môi trường."
    )

spark = (
    SparkSession.builder
    .appName("EcommerceSilverLayer")
    .master("local[*]")

    .config("spark.jars.packages", PACKAGES)

    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.lakehouse",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.lakehouse.type",
        "rest"
    )
    .config(
        "spark.sql.catalog.lakehouse.uri",
        "http://iceberg-rest:8181"
    )

    .config(
        "spark.sql.catalog.lakehouse.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.sql.catalog.lakehouse.s3.path-style-access",
        "true"
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        AWS_ACCESS_KEY_ID
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        AWS_SECRET_ACCESS_KEY
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .getOrCreate()
)

print("✅ SparkSession created")
print("Spark version:", spark.version)


## 6.2. Kiểm tra Bronze


In [ ]:
spark.sql("SHOW TABLES IN lakehouse.bronze").show(truncate=False)


## 6.3. Đọc các bảng Bronze


In [ ]:
customers_bronze = spark.table("lakehouse.bronze.customers")
products_bronze = spark.table("lakehouse.bronze.products")
orders_bronze = spark.table("lakehouse.bronze.orders")
order_items_bronze = spark.table("lakehouse.bronze.order_items")
payments_bronze = spark.table("lakehouse.bronze.payments")
reviews_bronze = spark.table("lakehouse.bronze.reviews")

bronze_tables = {
    "customers": customers_bronze,
    "products": products_bronze,
    "orders": orders_bronze,
    "order_items": order_items_bronze,
    "payments": payments_bronze,
    "reviews": reviews_bronze
}

for name, df in bronze_tables.items():
    print(f"{name:15} {df.count():,} rows")


## 6.4. Customers


In [ ]:
customers_silver = (
    customers_bronze
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

(
    customers_silver
    .writeTo("lakehouse.silver.customers")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.customers")


## 6.5. Products


In [ ]:
products_silver = (
    products_bronze
    .filter(F.col("product_id").isNotNull())
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_category_name",
        F.lower(F.trim(F.col("product_category_name")))
    )
)

for column in [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]:
    if column in products_silver.columns:
        products_silver = products_silver.filter(
            F.col(column).isNull()
            | (F.col(column) >= 0)
        )

products_silver = products_silver.withColumn(
    "_processed_at",
    F.current_timestamp()
)

(
    products_silver
    .writeTo("lakehouse.silver.products")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.products")


## 6.6. Orders


In [ ]:
orders_silver = (
    orders_bronze
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["order_id"])
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
)

for column in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]:
    if column in orders_silver.columns:
        orders_silver = orders_silver.withColumn(
            column,
            F.to_timestamp(F.col(column))
        )

orders_silver = orders_silver.withColumn(
    "_processed_at",
    F.current_timestamp()
)

(
    orders_silver
    .writeTo("lakehouse.silver.orders")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.orders")


## 6.7. Order Items


In [ ]:
order_items_silver = (
    order_items_bronze
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_item_id").isNotNull())
    .dropDuplicates(["order_id", "order_item_id"])
    .withColumn(
        "shipping_limit_date",
        F.to_timestamp(F.col("shipping_limit_date"))
    )
    .filter(
        F.col("price").isNull()
        | (F.col("price") >= 0)
    )
    .filter(
        F.col("freight_value").isNull()
        | (F.col("freight_value") >= 0)
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

(
    order_items_silver
    .writeTo("lakehouse.silver.order_items")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.order_items")


## 6.8. Payments


In [ ]:
payments_silver = (
    payments_bronze
    .filter(F.col("order_id").isNotNull())
    .filter(
        F.col("payment_installments").isNotNull()
        & (F.col("payment_installments") > 0)
    )
    .filter(F.col("payment_value") >= 0)
    .dropDuplicates(["order_id", "payment_sequential"])
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )
    .withColumn(
        "_processed_at",
        F.current_timestamp()
    )
)

(
    payments_silver
    .writeTo("lakehouse.silver.payments")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.payments")


### Kiểm tra payments bị loại


In [ ]:
payments_removed_by_filter = (
    payments_bronze
    .filter(
        F.col("order_id").isNull()
        | F.col("payment_installments").isNull()
        | (F.col("payment_installments") <= 0)
        | (F.col("payment_value") < 0)
    )
)

print(
    "Rows removed by filter:",
    payments_removed_by_filter.count()
)

payments_removed_by_filter.show(
    20,
    truncate=False
)


## 6.9. Reviews


In [ ]:
reviews_silver = (
    reviews_bronze
    .filter(F.col("review_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
    .dropDuplicates(["review_id", "order_id"])
    .filter(F.col("review_score").between(1, 5))
)

for column in [
    "review_creation_date",
    "review_answer_timestamp"
]:
    if column in reviews_silver.columns:
        reviews_silver = reviews_silver.withColumn(
            column,
            F.to_timestamp(F.col(column))
        )

for column in [
    "review_comment_title",
    "review_comment_message"
]:
    if column in reviews_silver.columns:
        reviews_silver = reviews_silver.withColumn(
            column,
            F.trim(F.col(column))
        )

reviews_silver = reviews_silver.withColumn(
    "_processed_at",
    F.current_timestamp()
)

(
    reviews_silver
    .writeTo("lakehouse.silver.reviews")
    .using("iceberg")
    .createOrReplace()
)

print("✅ silver.reviews")


## 6.10. So sánh Bronze và Silver


In [ ]:
silver_tables = {
    "customers": spark.table("lakehouse.silver.customers"),
    "products": spark.table("lakehouse.silver.products"),
    "orders": spark.table("lakehouse.silver.orders"),
    "order_items": spark.table("lakehouse.silver.order_items"),
    "payments": spark.table("lakehouse.silver.payments"),
    "reviews": spark.table("lakehouse.silver.reviews")
}

rows = []

for name in bronze_tables:
    bronze_count = bronze_tables[name].count()
    silver_count = silver_tables[name].count()

    rows.append(
        (
            name,
            bronze_count,
            silver_count,
            bronze_count - silver_count
        )
    )

comparison_df = spark.createDataFrame(
    rows,
    [
        "table_name",
        "bronze_rows",
        "silver_rows",
        "removed_rows"
    ]
)

comparison_df.show(truncate=False)


## 6.11. Kiểm tra key


In [ ]:
def check_key_quality(
    df,
    table_name,
    key_columns
):
    condition = None

    for column in key_columns:
        current = F.col(column).isNull()

        condition = (
            current
            if condition is None
            else condition | current
        )

    null_count = (
        df
        .filter(condition)
        .count()
    )

    duplicate_groups = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name:15} "
        f"| NULL key rows: {null_count:5} "
        f"| duplicate key groups: {duplicate_groups:5}"
    )

check_key_quality(
    silver_tables["customers"],
    "customers",
    ["customer_id"]
)

check_key_quality(
    silver_tables["products"],
    "products",
    ["product_id"]
)

check_key_quality(
    silver_tables["orders"],
    "orders",
    ["order_id"]
)

check_key_quality(
    silver_tables["order_items"],
    "order_items",
    ["order_id", "order_item_id"]
)

check_key_quality(
    silver_tables["payments"],
    "payments",
    ["order_id", "payment_sequential"]
)

check_key_quality(
    silver_tables["reviews"],
    "reviews",
    ["review_id", "order_id"]
)


## 6.12. Foreign-key checks


In [ ]:
customers_s = silver_tables["customers"]
products_s = silver_tables["products"]
orders_s = silver_tables["orders"]
order_items_s = silver_tables["order_items"]
payments_s = silver_tables["payments"]
reviews_s = silver_tables["reviews"]

fk_orders_customers = (
    orders_s.alias("o")
    .join(
        customers_s.alias("c"),
        F.col("o.customer_id")
        == F.col("c.customer_id"),
        "left_anti"
    )
)

fk_items_orders = (
    order_items_s.alias("oi")
    .join(
        orders_s.alias("o"),
        F.col("oi.order_id")
        == F.col("o.order_id"),
        "left_anti"
    )
)

fk_items_products = (
    order_items_s.alias("oi")
    .join(
        products_s.alias("p"),
        F.col("oi.product_id")
        == F.col("p.product_id"),
        "left_anti"
    )
)

fk_payments_orders = (
    payments_s.alias("p")
    .join(
        orders_s.alias("o"),
        F.col("p.order_id")
        == F.col("o.order_id"),
        "left_anti"
    )
)

fk_reviews_orders = (
    reviews_s.alias("r")
    .join(
        orders_s.alias("o"),
        F.col("r.order_id")
        == F.col("o.order_id"),
        "left_anti"
    )
)

print(
    "orders.customer_id -> customers.customer_id:",
    fk_orders_customers.count()
)

print(
    "order_items.order_id -> orders.order_id:",
    fk_items_orders.count()
)

print(
    "order_items.product_id -> products.product_id:",
    fk_items_products.count()
)

print(
    "payments.order_id -> orders.order_id:",
    fk_payments_orders.count()
)

print(
    "reviews.order_id -> orders.order_id:",
    fk_reviews_orders.count()
)


## 6.13. Kiểm tra Silver tables


In [ ]:
spark.sql("SHOW TABLES IN lakehouse.silver").show(truncate=False)


## 6.14. Iceberg snapshot


In [ ]:
spark.sql("""
SELECT
    committed_at,
    snapshot_id,
    operation
FROM lakehouse.silver.orders.snapshots
ORDER BY committed_at DESC
""").show(truncate=False)


## 6.15. Hoàn tất


In [ ]:
expected = {
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "reviews"
}

actual = {
    row.tableName
    for row in spark.sql(
        "SHOW TABLES IN lakehouse.silver"
    ).collect()
}

print("========== SILVER LAYER ==========")

for table in sorted(expected):
    if table in actual:
        print(f"✅ {table}")
    else:
        print(f"❌ {table}")

print("✅ Step 06 hoàn tất")
